In [11]:
import sys
# 현재 주피터 노트북이 실제로 실행되고 있는 파이썬 가상환경 경로에 uv로 직접 설치합니다.
!{sys.executable} -m pip install selenium beautifulsoup4 requests

c:\Users\student\p1-data\c3-deep-leanring\.venv\Scripts\python.exe: No module named pip


In [13]:
import os
import re
import requests

def quick_google_crawl(search_keyword, save_path, max_images=100):
    """브라우저 제어 없이 구글 소스에서 고화질 웹 이미지 주소를 직접 파싱하여 다운로드"""
    os.makedirs(save_path, exist_ok=True)
    
    # 구글 서버를 속이기 위한 최소한의 유저 에이전트 설정
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    # 구글 이미지 검색 URL
    url = f"https://www.google.com/search?q={search_keyword}&tbm=isch"
    
    print(f"\n🔎 '{search_keyword}' 데이터 수집 시작...")
    try:
        response = requests.get(url, headers=headers, timeout=10)
        html_content = response.text
    except Exception as e:
        print(f"❌ 구글 서버 연결 실패: {e}")
        return

    # 구글 소스 내부에 숨겨진 실제 원본 이미지 URL(주로 주소 뒤에 jpg, png 등이 붙은 형태) 정규식으로 추출
    # 자주 바뀌는 HTML 태그나 클래스명 구조에 전혀 영향을 받지 않는 가장 확실한 방법입니다.
    img_urls = re.findall(r'(https?://[^\s"\';<>]+?\.(?:jpg|jpeg|png))', html_content)
    
    # 중복 주소 제거
    img_urls = list(set(img_urls))
    print(f"📸 유효한 이미지 주소 {len(img_urls)}개 분석 완료. 다운로드 시작...")

    count = 0
    for img_url in img_urls:
        if count >= max_images:
            break
            
        # 암호화된 구글 자체 썸네일 주소는 화질이 깨지므로 패스
        if "encrypted" in img_url or "gstatic" in img_url:
            continue
            
        try:
            img_res = requests.get(img_url, headers=headers, timeout=5)
            if img_res.status_code == 200:
                file_name = f"{search_keyword}_{count}.jpg"
                full_path = os.path.join(save_path, file_name)
                
                with open(full_path, 'wb') as f:
                    f.write(img_res.content)
                count += 1
        except:
            # 폭파된 사이트나 링크 깨진 주소는 쿨하게 넘김
            continue

    print(f"📂 저장 완료: '{save_path}' 폴더에 {count}개의 사진이 저장되었습니다.")

# ==================== 실행 영역 ==================== #
if __name__ == "__main__":
    # 데이터셋 저장 폴더 정의
    TRAIN_HUMAN = "dataset/train/human"
    TRAIN_MANNEQUIN = "dataset/train/mannequin"
    
    # 별도 설정 없이 바로 실행 (각각 60장 타겟, 필요시 숫자 조절 가능)
    quick_google_crawl(search_keyword="사람 피부 손 사진", save_path=TRAIN_HUMAN, max_images=60)
    quick_google_crawl(search_keyword="의류 매장 마네킹 사진", save_path=TRAIN_MANNEQUIN, max_images=60)
    
    print("\n🎉 데이터셋 폴더(`dataset/`)를 확인해 보세요! 깔끔하게 정렬되어 들어가 있습니다.")


🔎 '사람 피부 손 사진' 데이터 수집 시작...
📸 유효한 이미지 주소 0개 분석 완료. 다운로드 시작...
📂 저장 완료: 'dataset/train/human' 폴더에 0개의 사진이 저장되었습니다.

🔎 '의류 매장 마네킹 사진' 데이터 수집 시작...
📸 유효한 이미지 주소 0개 분석 완료. 다운로드 시작...
📂 저장 완료: 'dataset/train/mannequin' 폴더에 0개의 사진이 저장되었습니다.

🎉 데이터셋 폴더(`dataset/`)를 확인해 보세요! 깔끔하게 정렬되어 들어가 있습니다.
